# KrishiDisha assistant - QLoRA fine-tuning on Kaggle

Trains the two candidate bases (Qwen2.5-3B-Instruct, Gemma-3-4B-it) on `data/llm/train.jsonl` with Unsloth + TRL, then merges and exports GGUF for Ollama.

**Setup**: Accelerator GPU T4 x2 (or P100), Internet ON. Upload `data/llm/train.jsonl` + `eval.jsonl` as a private Kaggle dataset and attach it. For Gemma accept the licence on Hugging Face and add `HF_TOKEN` as a Kaggle secret.

One epoch of ~30k examples is roughly 3.5-4.5 h on a T4 for the 3B model; keep `--save-steps` so a session that dies resumes with `--resume`.

In [ ]:
# Source comes from the private Kaggle dataset (git archive of the lab repo), not from GitHub.
# Kaggle extracts uploaded zips, so the tree is at .../krishidisha-lab-src/src/krishidisha; fall back to src.zip.
import glob, os, shutil, zipfile
shutil.rmtree("/tmp/krishidisha", ignore_errors=True)
_dirs = glob.glob("/kaggle/input/**/krishidisha-lab-src/src/krishidisha", recursive=True)
if _dirs:
    shutil.copytree(_dirs[0], "/tmp/krishidisha")
    _ver = os.path.join(os.path.dirname(os.path.dirname(_dirs[0])), "VERSION.txt")
else:
    _zip = glob.glob("/kaggle/input/**/krishidisha-lab-src/**/src.zip", recursive=True)[0]
    zipfile.ZipFile(_zip).extractall("/tmp")
    _ver = os.path.join(os.path.dirname(_zip), "VERSION.txt")
print("source version:", open(_ver).read().strip() if os.path.exists(_ver) else "unknown", "->", os.listdir("/tmp/krishidisha")[:8])
BASE = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"      # or unsloth/gemma-3-4b-it-unsloth-bnb-4bit
MERGE_BASE = "Qwen/Qwen2.5-3B-Instruct"           # unquantized base for the merge (google/gemma-3-4b-it)
EPOCHS = 2
SMOKE = True   # True: setup + 200-example smoke run only (~10 min). Set False for the real run + GGUF export.

import os, subprocess, sys, glob, traceback
W = "/kaggle/working"
LOG = open(f"{W}/setup.log", "a")
def log(*a):
    print(*a); print(*a, file=LOG); LOG.flush()
try:
    log("input tree:", subprocess.run(["bash", "-c", "find /kaggle/input -maxdepth 3 | head -40"], capture_output=True, text=True).stdout)
    log("gpu:", subprocess.run(["bash", "-c", "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo no-gpu"], capture_output=True, text=True).stdout)
    os.chdir("/tmp/krishidisha")
    train = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0]
    evalf = glob.glob("/kaggle/input/**/eval.jsonl", recursive=True)[0]
    log(train, evalf, "smoke" if SMOKE else "full")
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        log("no HF_TOKEN secret (fine for Qwen):", e)
except Exception:
    log(traceback.format_exc()); raise


In [ ]:
CMD = f"pip install -q unsloth trl peft bitsandbytes datasets pyyaml 2>&1 | tail -5 | tee {W}/pip.log"
!{CMD}
CMD = f"python -c 'import torch, unsloth, trl; print(torch.__version__, torch.cuda.is_available(), unsloth.__version__, trl.__version__)' 2>&1 | tee -a {W}/pip.log"
!{CMD}


In [ ]:
# smoke test first: 200 examples, 0.05 epoch
CMD = f"python -m ml.llm.train_llm --data {train} --eval {evalf} --base {BASE} --out {W}/kd-smoke --epochs 0.05 --limit 200 2>&1 | tee {W}/smoke.log | tail -30"
!{CMD}


In [ ]:
OUT = f"{W}/kd-" + ("gemma" if "gemma" in BASE.lower() else "qwen")
if not SMOKE:
    CMD = f"python -m ml.llm.train_llm --data {train} --eval {evalf} --base {BASE} --out {OUT} --epochs {EPOCHS} --save-steps 200 --resume 2>&1 | tee {W}/train.log | tail -40"
    !{CMD}
    print(open(f"{OUT}/sample_generations.md").read()[:4000])


In [ ]:
# Merge + GGUF (needs llama.cpp; ~10 min). Download the GGUF and Modelfile from the Output tab, then locally:
#   ollama create krishidisha -f models/llm/Modelfile.qwen
if not SMOKE:
    # do NOT pip install llama.cpp's requirements file: it downgrades transformers and breaks peft; the converter only needs gguf + sentencepiece
    CMD = f"git clone --depth 1 https://github.com/ggerganov/llama.cpp {W}/llama.cpp 2>&1 | tail -2 && pip install -q gguf sentencepiece protobuf 2>&1 | tail -2"
    !{CMD}
    CMD = f"cd {W}/llama.cpp && cmake -B build -DGGML_CUDA=OFF > /dev/null && cmake --build build --target llama-quantize -j > /dev/null"
    !{CMD}
    CMD = f"python -m ml.llm.export_gguf --adapter {OUT}/adapter --base {MERGE_BASE} --out {W}/gguf --quant q4_k_m q8_0 --llama-cpp {W}/llama.cpp 2>&1 | tee {W}/export.log | tail -20"
    !{CMD}
    CMD = f"ls -la {W}/gguf"
    !{CMD}


In [ ]:
import shutil; shutil.rmtree(f"{W}/llama.cpp", ignore_errors=True)
